In [ ]:
# Step 1: Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, precision_score, recall_score,
                             f1_score)
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Step 2: Load data and create a 10% balanced sample
df = pd.read_csv('fraudData-Feature-Engineered.csv')
print(f"Original dataset shape: {df.shape}")
print(f"Original class distribution:\n{df['is_fraud'].value_counts()}\n")

fraud = df[df['is_fraud'] == 1].sample(frac=1, random_state=42)
non_fraud = df[df['is_fraud'] == 0].sample(frac=1, random_state=42)

# Combine and shuffle the sampled data
df_sampled = pd.concat([fraud, non_fraud]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Sampled dataset shape: {df_sampled.shape}")
print(f"Sampled class distribution:\n{df_sampled['is_fraud'].value_counts()}")
print(f"Fraud ratio: {df_sampled['is_fraud'].mean():.4f}")

In [ ]:
# Step 3: Define features and perform train-test split
features = ['amt', 'hour', 'day_of_week', 'is_weekend', 'distance',
            'transactions_per_card', 'avg_amt_per_card',
            'unique_merchants_per_card', 'age',
            'merchant_fraud_rate', 'merchant_avg_amt', 'merchant_txn_count',
            'category_fraud_rate', 'category_count']

X = df_sampled[features]
y = df_sampled['is_fraud']

# 80/20 stratified split to preserve class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"Train fraud ratio: {y_train.mean():.4f}")
print(f"Test fraud ratio:  {y_test.mean():.4f}")

In [ ]:
# Step 4: Train Random Forest with balanced class weights
model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1  
)
model.fit(X_train, y_train)

# Generate predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Random Forest trained successfully!")

In [ ]:
# Step 5: Evaluate the model
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("=" * 50)
print("RANDOM FOREST — Evaluation Metrics")
print("=" * 50)
print(f"Precision:  {precision:.4f}")
print(f"Recall:     {recall:.4f}")
print(f"F1-Score:   {f1:.4f}")
print(f"ROC-AUC:    {roc_auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Non-Fraud', 'Fraud']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Non-Fraud', 'Fraud'],
            yticklabels=['Non-Fraud', 'Fraud'])
plt.title('Random Forest — Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:

# Step 6: Save the trained model and preprocessing artifacts for deployment

import joblib

# Save the Random Forest model
joblib.dump(model, 'rf_model.pkl')
print("Saved: rf_model.pkl")

# Save as the canonical 'best model' for the Streamlit app
joblib.dump(model, 'fraud_model.pkl')
print("Saved: fraud_model.pkl  (main deployment model)")

# Save the feature list used during training
# Keeping this in sync with training prevents silent feature-mismatch bugs.
joblib.dump(features, 'features.pkl')
print("Saved: features.pkl")

# Note on scaler
# Random Forest does not require feature scaling, so no scaler.pkl
print("\nAll deployment artifacts saved successfully.")
print(f"  Model        : {model.__class__.__name__}")
print(f"  Features ({len(features)}): {features}")
